# Building ANN with Pytorch on Entire 60,000 images using GPU

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

In [2]:
torch.manual_seed(42)

In [7]:
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu') 
print(f"Using device: {device}")

Using device: mps


In [9]:
df = pd.read_csv('data/fashion-mnist_train.csv')
df.head(2)

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,pixel10,pixel11,pixel12,pixel13,pixel14,pixel15,pixel16,pixel17,pixel18,pixel19,pixel20,pixel21,pixel22,pixel23,pixel24,pixel25,pixel26,pixel27,pixel28,pixel29,pixel30,pixel31,pixel32,pixel33,pixel34,pixel35,pixel36,pixel37,pixel38,pixel39,...,pixel745,pixel746,pixel747,pixel748,pixel749,pixel750,pixel751,pixel752,pixel753,pixel754,pixel755,pixel756,pixel757,pixel758,pixel759,pixel760,pixel761,pixel762,pixel763,pixel764,pixel765,pixel766,pixel767,pixel768,pixel769,pixel770,pixel771,pixel772,pixel773,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [10]:
X = df.iloc[:,1:].values
y = df.iloc[:,0].values

In [11]:
#train test split
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

In [12]:
#scaling the values
X_train = X_train/255.0
X_test = X_test/255.0

In [13]:
#Create custom dataset

class CustomDataset(Dataset):
    def __init__(self,features,label):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.label = torch.tensor(label, dtype=torch.long)

    def __len__(self):
        return self.features.shape[0]
    
    def __getitem__(self, index):
        return self.features[index], self.label[index]

In [14]:
#create train dataset
train_dataset = CustomDataset(X_train, y_train)

In [15]:
#create test dataset
test_dataset = CustomDataset(X_test,y_test)

In [16]:
#create train and test loader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [17]:
#define nn class
class Mynn(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(num_features,128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64,10)
        )

    def forward(self,x):
        return self.model(x)

In [18]:
#set learning rate and epochs
epochs = 100
learning_rate = 0.1

In [19]:
#instantiate the model
model = Mynn(X_train.shape[1])
model = model.to(device)
#loss function 
criterion = nn.CrossEntropyLoss()

#optimizer
optimizer = optim.SGD(model.parameters(), learning_rate)

In [20]:
#training loop

for epoch in range(epochs):
    total_epoch_loss = 0
    for batch_features, batch_label in train_loader:
        batch_features, batch_label = batch_features.to(device), batch_label.to(device)
        #forwardpass
        outputs = model(batch_features)
        #calculate loss
        loss = criterion(outputs, batch_label)
        #backpass
        optimizer.zero_grad()
        loss.backward()
        #updateparams
        optimizer.step()

        total_epoch_loss = total_epoch_loss + loss.item()

    avg_loss = total_epoch_loss/len(train_loader)
    print(f'Epoch: {epoch+1}, Loss:{avg_loss:.4f}')
    

Epoch: 1, Loss:0.6353
Epoch: 2, Loss:0.4308
Epoch: 3, Loss:0.3867
Epoch: 4, Loss:0.3579
Epoch: 5, Loss:0.3368
Epoch: 6, Loss:0.3220
Epoch: 7, Loss:0.3072
Epoch: 8, Loss:0.2955
Epoch: 9, Loss:0.2857
Epoch: 10, Loss:0.2740
Epoch: 11, Loss:0.2681
Epoch: 12, Loss:0.2565
Epoch: 13, Loss:0.2508
Epoch: 14, Loss:0.2448
Epoch: 15, Loss:0.2398
Epoch: 16, Loss:0.2306
Epoch: 17, Loss:0.2252
Epoch: 18, Loss:0.2223
Epoch: 19, Loss:0.2141
Epoch: 20, Loss:0.2099
Epoch: 21, Loss:0.2059
Epoch: 22, Loss:0.2035
Epoch: 23, Loss:0.1973
Epoch: 24, Loss:0.1927
Epoch: 25, Loss:0.1900
Epoch: 26, Loss:0.1854
Epoch: 27, Loss:0.1807
Epoch: 28, Loss:0.1756
Epoch: 29, Loss:0.1720
Epoch: 30, Loss:0.1716
Epoch: 31, Loss:0.1673
Epoch: 32, Loss:0.1647
Epoch: 33, Loss:0.1599
Epoch: 34, Loss:0.1594
Epoch: 35, Loss:0.1592
Epoch: 36, Loss:0.1552
Epoch: 37, Loss:0.1484
Epoch: 38, Loss:0.1494
Epoch: 39, Loss:0.1445
Epoch: 40, Loss:0.1415
Epoch: 41, Loss:0.1365
Epoch: 42, Loss:0.1374
Epoch: 43, Loss:0.1355
Epoch: 44, Loss:0.13

In [21]:
#set model to eval mode

model.eval()

Mynn(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [22]:
#evaluation code

total = 0
correct = 0

with torch.no_grad():
    for batch_features, batch_label in test_loader:
        batch_features, batch_label = batch_features.to(device), batch_label.to(device)
        output = model(batch_features)
        _, predicted = torch.max(output,1)
        
        total = total + batch_label.shape[0]
        correct = correct + (predicted == batch_label).sum().item()
    
print(correct/total)


0.8836666666666667
